# [기초-실습] 통계 101×데이터 분석: (4장) 추론통계~신뢰구간

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비 — 한글 폰트 설치 및 라이브러리 불러오기

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

# 문제 1. 표본조사 체험하기

**📘 문제**

- 온라인 쇼핑몰은 전체 고객 수가 너무 많아, 모든 고객을 조사하기 어렵습니다.

- 그래서 무작위로 고객 30명을 뽑아 평균 만족도를 계산하고 이를 전체 만족도의 추정값으로 사용하려 합니다.

- 이번 실습에서는 직접 표본을 뽑고, 표본 평균을 구해보며,
  **“표본마다 결과가 달라질 수 있다”**는 추론 통계의 핵심 개념을 체험해봅니다.

In [ ]:
# 모집단 생성 (전체 고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.5, scale=1.2, size=10000)
population = np.clip(population, 1, 10)  # 1점 ~ 10점 사이로 제한
df_pop = pd.DataFrame({'score': population})

# 전체 모집단 시각화
sns.histplot(df_pop['score'], bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 표본을 무작위로 여러 번 뽑아 보고, 표본 평균이 어떻게 변하는지 확인해봅시다.

- 히스토그램을 그리고, 표본 평균의 분포 형태를 관찰해봅시다.

In [ ]:
# [문제 1] Q1. 모집단에서 무작위로 30명을 뽑아 표본 평균을 구해봅시다.

# 모집단에서 30명을 무작위 추출
sample_30 = df_pop.sample(n=30, random_state=2025)

# 표본 평균 계산
sample_mean = sample_30["score"].mean()

print("표본 데이터:")
print(sample_30)

print("표본 평균:", sample_mean)

In [ ]:
# [문제 1] Q2. 이 과정을 500번 반복하고, 표본 평균을 리스트에 저장합니다.

result = []

for i in range (500):
    sample_30 = df_pop.sample(n=30, random_state=2025 + i)
    sample_mean = sample_30["score"].mean()
    result.append(sample_mean)
    result_mean = np.mean(result)
print(result_mean)

In [ ]:
# [문제 1] Q3. 표본 평균들의 분포를 히스토그램으로 그려봅시다. 평균선도 함께 표시해 봅시다.
# 여기에 코드를 작성해주세요.

sns.histplot(result, bins=40, kde=True)
plt.axvline(
    result_mean,
    linestyle="--",
    linewidth=2,
    label=f"평균: {result_mean:.2f}"
)
plt.title("표본평균 분포")
plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 평균들은 어떤 값 주변에 많이 분포해 있나요? 이 값은 전체 모집단 평균과 얼마나 비슷한가요?

- 표본을 1번 뽑았을 때와 500번을 반복해서 뽑았을 때, 표본 평균의 분포나 신뢰성에는 어떤 차이가 있나요

- 친구가 다른 표본을 뽑았다면 같은 평균이 나왔을까요? 비슷한 결과가 나왔더라도 완전히 같지 않았다면, 그 이유는 무엇일까요?

- 표본 평균들의 분포는 어떤 모양인가요? 종 모양의 정규분포처럼 보이나요? 그렇다면 왜 그렇게 되는 걸까요?

In [ ]:
# [문제 1] 데이터를 어떻게 읽을까요?
# 전체 모집단 평균과 대체적으로 비슷함.
# 표본을 1회 뽑았을 경우, 그 값이 모집단 평균과 얼마나 차이가 날 수 있는지 판단하기 어려움. 충분한 표본이 쌓이면 표본이 어느 주변에 분포하는지 알 수 있고, 이로 인해 모집단을 추정할 수 있음. 
# 완전히 같을 가능성은 낮지만, 같은 모집단에서 같은 수인 30명을 무작위로 뽑았으므로, 친구의 표본 평균도 모집단 평균인 7.48점 주변에서 나올 가능성이 높음.
# 다소 오차가 있지만 정규분포의 형태를 띄고 있음. 표본 평균 데이터가 많을수록 극단값들이 상쇄되며 정규분포의 형태에 가까워지는 성질이 있음.

# 문제 2. 중심극한정리

**📘 문제**

- 현실에서는 모집단의 분포가 정규분포가 아닐 수도 있습니다.

- 예를 들어, 일부 고객은 매우 높은 점수를 주고, 대부분은 낮은 점수를 주는 만족도 분포가 있을 수 있죠. (예: 지수분포)

- 이처럼 원래 분포가 비정규분포여도,
  표본을 여러 번 뽑아 평균을 계산하면, 그 평균들의 분포는 정규분포에 가까워진다는 것을
  **중심극한정리(Central Limit Theorem)**라고 합니다.

- 이번 실습에서는 다양한 크기의 표본을 뽑아 평균을 계산하고,
  그 평균들의 분포가 어떻게 변하는지를 직접 실험해 봅니다.

In [ ]:
# 지수분포를 따르는 모집단 생성

from scipy.stats import skew

np.random.seed(2025)
population = np.random.exponential(scale=50, size=100000)  # 평균 50, 비대칭 분포

# 모집단 시각화
sns.histplot(population, bins=50, kde=True)
plt.title("고객 구매 금액 분포 (모집단: 지수분포)")
plt.xlabel("구매 금액")
plt.show()

**📌 아래를 수행해 보세요:**

- 비대칭적인 모집단(지수분포)에서 무작위로 표본을 추출해 평균을 구해봅시다.

- 표본 크기를 바꿔가며, 표본 평균들의 분포가 어떻게 변화하는지 확인해봅시다.

- 히스토그램을 그리고, 분포의 모양을 관찰해봅시다.

- 표본 크기가 커질수록 표본 평균 분포의 모양과 **퍼진 정도(분산)**가 어떻게 변하는지 관찰해봅시다.

In [ ]:
# [문제 2] Q1. 모집단에서 표본을 1000번 뽑고, 각 표본의 평균을 구해봅시다.
# 표본 크기 = 5일 때

sample_rng = np.random.default_rng(2026)

sample_sizes = [5]
sample_means_dict = {}

for sample_size in sample_sizes:
    means = []

    for _ in range(1000):
        sample = sample_rng.choice(
            population,
            size=sample_size,
            replace=False
        )

        means.append(sample.mean())

    means = np.array(means)
    sample_means_dict[sample_size] = means

    print(f"\n[표본 크기: {sample_size}]")
    print("표본평균 개수:", len(means))
    print("처음 5개의 표본평균:", means[:5])
    print(f"표본평균들의 평균: {means.mean():.3f}")
    print(f"표본평균 분포의 왜도: {skew(means, bias=False):.3f}")


In [ ]:
# [문제 2] Q2. 위 과정을 표본 크기 30, 100일 때도 반복해봅시다.

sample_rng = np.random.default_rng(2026)

sample_sizes = [5, 30, 100]
sample_means_dict = {}

for sample_size in sample_sizes:
    means = []

    for _ in range(1000):
        sample = sample_rng.choice(
            population,
            size=sample_size,
            replace=False
        )

        means.append(sample.mean())

    means = np.array(means)
    sample_means_dict[sample_size] = means

    print(f"\n[표본 크기: {sample_size}]")
    print("표본평균 개수:", len(means))
    print("처음 5개의 표본평균:", means[:5])
    print(f"표본평균들의 평균: {means.mean():.3f}")
    print(f"표본평균 분포의 왜도: {skew(means, bias=False):.3f}")


In [ ]:
# [문제 2] Q3. 각 표본 크기별로 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 평균선을 함께 표시해 봅시다.

import matplotlib.pyplot as plt
import seaborn as sns

for sample_size in sample_sizes:
    sample_means = sample_means_dict[sample_size]

    mean_of_means = sample_means.mean()
    observed_skewness = skew(sample_means, bias=False)

    # 지수분포 표본평균의 이론적 왜도
    theoretical_skewness = 2 / np.sqrt(sample_size)

    print(f"\n[표본 크기 {sample_size}]")
    print(f"표본평균들의 평균: {mean_of_means:.3f}")
    print(f"실제 계산 왜도: {observed_skewness:.3f}")
    print(f"이론적 왜도: {theoretical_skewness:.3f}")

    plt.figure(figsize=(8, 5))

    sns.histplot(
        sample_means,
        bins=40,
        kde=True
    )

    plt.axvline(
        mean_of_means,
        linestyle="--",
        linewidth=2,
        label=f"평균: {mean_of_means:.2f}"
    )

    plt.title(
        f"표본 크기 {sample_size}의 표본평균 분포\n"
        f"왜도 = {observed_skewness:.3f}"
    )
    plt.xlabel("표본평균")
    plt.ylabel("빈도")
    plt.legend()
    plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을 때 (예: 5), 평균들의 분포는 어떤 모양인가요?

- 표본 크기가 커질수록 평균 분포의 모양은 어떤 변화를 보이나요?

- 원래 모집단은 비대칭이었는데, 왜 평균들의 분포는 정규분포처럼 바뀌었을까요?

- 이 실험을 통해 중심극한정리를 어떻게 이해하게 되었나요?

- 표본 크기에 따라 **분포의 넓이(흩어짐)**는 어떻게 달라지나요?

In [ ]:
# [문제 2]데이터를 어떻게 읽을까요?

# 표본 크기가 작을 때, 평균들의 분포는 뚜렷한 우편향의 형태임.
# 표본이 커질수록 왜도가 줄어들고, 대칭적인 정규분포에 가까워짐.
# 표본 데이터가 충분 많아질수록 모집단 평균에 가까워지는 현상임.
# 분포 역시 표준오차가 감소하여 표본평균들이 모집단 평균 주변에 더 좁게 모이므로 분포의 넓이도 감소함.

# 문제 3. 표준오차

**📘 문제**

- 앞선 실습에서 우리는 **표본 크기(n)가 커질수록 표본 평균들의 분포가 더 좁아진다**는 것을 확인했습니다.
- 이처럼 표본 평균들이 얼마나 흩어져 있는지(분포의 퍼진 정도)를 나타내는 값을 **표준오차(Standard Error, SE)**라고 부릅니다.
- 표준오차는 **표본 평균들의 표준편차**와 같은 의미이며, 이는 우리가 뽑은 표본 평균이 실제 모평균과 평균적으로 얼마나 떨어져 있을지를 나타내는 **'예상 오차의 크기'**입니다.

- 통계학적으로 이 표준오차는 **`SE = σ / √n`** (모집단 표준편차 / 표본 크기의 제곱근) 이라는 공식으로 계산할 수 있습니다.
- 이 공식은 **표본 크기(n)가 커질수록 표준오차(SE)가 작아진다**는 것을 명확히 보여줍니다.

- 이번 실습에서는 여러 크기의 표본을 뽑아, 시뮬레이션을 통해 얻은 **표본 평균들의 표준편차(실험값)**가 공식으로 계산한 **표준오차(이론값)**와 얼마나 일치하는지 직접 확인해봅니다.

In [ ]:
# 모집단 생성 (평균 100, 표준편차 15)
np.random.seed(2025)
population = np.random.normal(loc=100, scale=15, size=100000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 100, 표준편차 15)")
plt.xlabel("값")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 여러 크기의 표본(10, 30, 100, 500)을 각각 1000번 뽑고, 그 평균들을 구한 뒤, **표본 평균들의 표준편차(=실험적 표준오차)**를 계산해봅시다.

- 이 결과를 이론적인 표준오차 공식과 비교하는 표를 만들고, 시각화해봅시다.

In [ ]:
# [문제 3] Q1. 표본 크기 10, 30, 100, 500에 대해 각각 1000번 표본을 뽑고, 평균을 구해봅시다.
# 각 표본 평균 분포의 표준편차를 계산해봅시다.
# 결과를 리스트에 저장하고, 표로 정리해봅시다.

sample_sizes = [10, 30, 100, 500]
results = []

rng = np.random.default_rng(2025)

for sample_size in sample_sizes:
    sample_means = []

    for _ in range(1000):
        sample = rng.choice(
            population,
            size=sample_size,
            replace=False
        )
        sample_means.append(sample.mean())

    results.append({
        "표본 크기": sample_size,
        "표본 평균의 평균": np.mean(sample_means),
        "실험적 표준오차": np.std(sample_means, ddof=1),
        "이론적 표준오차": 15 / np.sqrt(sample_size)
    })

result_df = pd.DataFrame(results)
print(result_df)

In [ ]:
# [문제 3] Q2. 이론적인 표준오차와 비교해봅시다.
# [공식] 표준오차(SE) = 모집단 표준편차 / √표본크기

result_df["이론적 표준오차"] = 15 / np.sqrt(result_df["표본 크기"])

result_df["차이"] = (
    result_df["실험적 표준오차"]
    - result_df["이론적 표준오차"]
)

print(result_df)

In [ ]:
# [문제 3] Q3. 실험값과 이론값을 시각화해봅시다.
# 표본 크기를 x축, 표준오차를 y축으로 한 꺾은선 그래프를 그려봅시다.

!apt-get -qq install fonts-nanum

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

plt.plot(result_df["표본 크기"], result_df["실험적 표준오차"],
         marker="o", label="실험값")

plt.plot(result_df["표본 크기"], result_df["이론적 표준오차"],
         marker="x", linestyle="--", label="이론값")

plt.xlabel("표본 크기")
plt.ylabel("표준오차")
plt.title("실험적·이론적 표준오차 비교")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을수록, 표본 평균의 분포는 어떤 모양인가요? 넓게 퍼져 있나요?

- 표본 크기가 커질수록, 평균 분포는 어떻게 변하나요?

- 실험값과 이론값(공식 계산값)은 얼마나 비슷한가요?

- 왜 표본 크기가 커질수록 표준오차는 작아질까요?

In [ ]:
# [문제 3] 데이터를 어떻게 읽을까요?

# 표본 크기가 작을수록 표본 평균의 분포는 더 넓게 퍼져있으며, 크기가 커질수록 줄어듦.
# 실험적 표준오차와 이론적 표준오차의 차이는 모든 표본 크기에서 약 0.05보다 작아 매우 비슷함.
# 표본 크기가 커질수록 관측값의 데이터가 누적되며 개별 관측값의 무작위 변동이 서로 상쇄됨에 따라 표준오차가 감소함.

# 문제 4. 신뢰구간 계산과 해석

**📘 문제**

- 표본 평균은 모집단 평균을 추정하는 좋은 점 추정(Point Estimation) 값이지만, 표본오차 때문에 정확히 일치하지는 않습니다.

- 그래서 우리는 "모집단 평균이 아마 이 범위 안에 있을 것이다"라고 **구간으로 추정(Interval Estimation)**하는 것이 더 합리적입니다. 이때 사용하는 개념이 바로 **신뢰구간(Confidence Interval)**입니다.

- 신뢰구간은 표본평균 ± 오차범위 형태로 계산되며, 이 오차범위는 신뢰수준(예: 95%, 99%)과 표본오차에 의해 결정됩니다.

- 이번 실습에서는 **모집단 표준편차(σ)를 알 때(z-분포)**와 **모를 때(t-분포)**의 신뢰구간을 각각 계산해보고, 신뢰수준에 따라 구간의 폭이 어떻게 변하는지 확인해봅니다.

In [ ]:
# 모집단 생성
np.random.seed(2025)
population = np.random.normal(loc=70, scale=10, size=10000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 70, 표준편차 10)")
plt.xlabel("점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 30명을 무작위로 뽑아 평균, 표준편차, 표준오차를 계산해보세요.

- 95% 신뢰구간을 z-분포와 t-분포를 각각 사용해서 계산해보세요.

- 신뢰수준을 바꿨을 때(90%, 99%) 신뢰구간이 어떻게 변하는지 확인해보세요.

In [ ]:
# [문제 4] Q1. 모집단에서 표본 30명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 표준오차도 함께 계산해보세요.

rng = np.random.default_rng(2025)

sample = rng.choice(population, size=30, replace=False)

sample_mean = sample.mean()
sample_std = sample.std(ddof=1)
standard_error = sample_std / np.sqrt(30)

print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)
print("표준오차:", standard_error)

In [ ]:
# [문제 4] Q2. 모집단의 표준편차를 알고 있다고 가정하고, z-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

z = 1.96
se = 10 / np.sqrt(30)

lower = sample_mean - z * se
upper = sample_mean + z * se

print("95% 신뢰구간:", (lower, upper))

In [ ]:
# [문제 4] Q3. 모집단의 표준편차를 모른다고 가정하고, 표본 표준편차와 t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

from scipy.stats import t

n = 30
t_value = t.ppf(0.975, df=n - 1)
se = sample_std / np.sqrt(n)

# t.ppf: 95% 신뢰구간은 양쪽 꼬리에 각각 2.5%씩 남기므로, 1−0.025 = 0.975
# 자유도는 30-1 = 29: 표본이 30명이고, 앞의 29개 편차가 정해지면 마지막 1개의 편차는 자동으로 결정됨.

lower = sample_mean - t_value * se
upper = sample_mean + t_value * se

print("95% 신뢰구간:", (lower, upper))

In [ ]:
# [문제 4] Q4. 신뢰수준을 90%, 99%로 바꿔가며 신뢰구간을 계산해보고, 그 폭을 비교해봅시다.

n = 30
t_value = t.ppf(0.95, df=n - 1)
se = sample_std / np.sqrt(n)

lower = sample_mean - t_value * se
upper = sample_mean + t_value * se

print("90% 신뢰구간:", (lower, upper))

n = 30
t_value = t.ppf(0.995, df=n - 1)
se = sample_std / np.sqrt(n)

lower = sample_mean - t_value * se
upper = sample_mean + t_value * se

print("99% 신뢰구간:", (lower, upper))

**🧠 데이터를 어떻게 읽을까요?**

- z-분포와 t-분포를 사용한 신뢰구간은 얼마나 차이가 있나요?

- 신뢰수준이 높아질수록 신뢰구간의 폭은 어떻게 변하나요? 왜 그럴까요?

- 신뢰구간이 넓다는 건 좋은 걸까요? 나쁜 걸까요?

- 이 데이터가 실제 고객 만족도라면, 신뢰구간 정보를 마케팅 전략에 어떻게 활용할 수 있을까요?

In [ ]:
# [문제 4] 데이터를 어떻게 읽을까요?

# z분포와 t-분포의 신뢰구간에 큰 차이는 없으나, 신뢰구간 범위는 z분포가 약소하게 넓음.
# 다만, 동일한 표준오차를 사용했을 경우 t-구간이 z구간보다 넓을 것임.
# t-분포는 모집단 표준편차를 모르기 때문에 발생하는 불확실성까지 반영하기 때문임.
# 신뢰구간이 넓다는 것은 모집단 평균의 추정이 덜 정밀하다는 의미이며, 신뢰구간이 좁은 경우 더 엄격한 검정이 가능함.
# 고객 만족도의 신뢰구간을 이용하면 전체 고객의 평균 만족도가 어느 범위에 있을지 추정 가능함.
# 신뢰구간의 하한이 목표 수준보다 높다면 높은 만족도를 마케팅 근거로 활용할 수 있음.
# 반대로 신뢰구간이 넓다면 조사 결과의 불확실성이 크다는 의미이므로, 표본을 더 확보한 뒤 전략을 결정하는 것이 적절함.

# 문제 5. 미니 프로젝트 - 고객 만족도 신뢰구간 추정

**📘 문제**

- 전체 고객 10,000명을 대상으로 만족도 조사를 하는 것은 시간과 비용이 많이 듭니다.
- 그래서 우리는 무작위로 일부 고객만 조사하여, 전체 고객의 평균 만족도를 추정하려 합니다.

- 이 프로젝트에서는 실제와 같은 상황을 가정하여, 표본을 뽑고 신뢰구간을 계산한 뒤, 이 결과를 바탕으로 마케팅 전략에 어떻게 활용할 수 있을지까지 생각해보는 실습을 진행합니다.

In [ ]:
# 모집단 생성 (고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.2, scale=1.0, size=10000)
population = np.clip(population, 1, 10)

# 모집단 시각화
sns.histplot(population, bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단을 생성하고, 거기서 표본을 40명 뽑아 평균을 계산해봅시다.

- 표본 평균과 표준편차를 바탕으로 95% 신뢰구간을 계산해봅시다.

- 히스토그램을 그리고 신뢰구간을 시각화해봅시다.

- 이 결과를 어떻게 해석하고, 마케팅 전략에 활용할 수 있을지 생각해봅시다.

In [ ]:
# [문제 5] Q1. 모집단에서 표본 40명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.

rng = np.random.default_rng(2025)

population_mean = population.mean()

print("모집단 평균:", population_mean)
print("신뢰구간 포함 여부:", lower <= population_mean <= upper)

sample = rng.choice(population, size=40, replace=False)

sample_mean = sample.mean()
sample_std = sample.std(ddof=1)

print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)

In [ ]:
# [문제 5] Q2. 표준오차(SE)를 구하고, t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

standard_error = sample_std / np.sqrt(40)
print("표준오차:", standard_error)

from scipy.stats import t

n = 40
t_value = t.ppf(0.975, df=n - 1)
se = sample_std / np.sqrt(n)

lower = sample_mean - t_value * se
upper = sample_mean + t_value * se

print("95% 신뢰구간:", (lower, upper))

In [ ]:
# [문제 5] Q3. 표본 데이터의 히스토그램을 그리고, 평균 및 신뢰구간을 함께 시각화해봅시다.

sns.histplot(sample, bins=10, kde=True)

plt.axvline(sample_mean, label="표본 평균")
plt.axvline(lower, linestyle="--", label="95% 신뢰구간")
plt.axvline(upper, linestyle="--")

plt.title("표본 고객 만족도 분포")
plt.xlabel("만족도 점수")
plt.legend()
plt.show()

In [ ]:
# [문제 5] Q4. 신뢰구간의 결과에 따라 어떤 구체적인 마케팅 전략을 세울 수 있을까요?

# 만족도 분포에서 95% 신뢰구간은 약 7점~7점 중반대로 나타남.
# 먼저 목표 만족도 점수 대비 신뢰구간 내 점수가 목표치에 도달하였는지 파악함.
# 목표 만족도보다 높다면 양호하다고 판단할 수 있을 것이고, 낮다면 개선이 필요한 지점임.
# 점수가 낮다면 개별 만족도 데이터를 확인하여 낮은 평가의 원인을 추가로 분석함.
# 만족도가 높은 집단을 대상으로 멤버십 또는 소정의 리워드, 재구매 혜택 등의 전략 활용이 가능함.
# 만족도가 낮은 집단에게는 쿠폰 등의 지급을 통해 재구매 독려와 함께 만족도를 높이기 위한 품질 개선을 병행하여 긍정적인 후기 빈도를 높일 수 있음.

**🧠 데이터를 어떻게 읽을까요?**

- 신뢰구간은 몇 점에서 몇 점 사이인가요?

- 이 구간은 전체 모집단 평균을 포함하고 있나요?

- 이 결과를 바탕으로 고객 만족도가 충분히 높다고 말할 수 있을까요?

- 만약 신뢰구간이 너무 넓게 나왔다면, 그 이유는 무엇이고 어떻게 개선할 수 있을까요?

In [ ]:
# [문제 5] 데이터를 어떻게 읽을까요?

# 신뢰구간은 약 7점에서 7.66점 사이이며, 전체 모집단 평균을 포함함.
# 다만 실제 조사에서는 모집단 평균을 알 수 없으므로, 신뢰구간이 실제 평균을 포함하는지 직접 확인은 불가능함. #GPT
# 10점 척도를 기준으로 보면 현재의 만족도 점수는 비교적 양호하다고 볼 수 있겠으나, 해당 데이터만으로 확신할 수는 없으며, 목표 만족도 점수 등과 비교해야 함.
# 신뢰구간이 넓게 나왔다면 표본 수가 적거나 오차범위가 크기 때문이며, 표본 수를 늘리거나 측정오차를 줄이는 등 개선의 여지가 있음.
# 신뢰수준을 낮춰 구간을 좁히는 것도 방법이 될 수 있지만, 모집단 평균을 포함하는 신뢰수준도 함께 낮아지는 것을 감안해야 함.